# BUSI Dataset Classification (Handling Class Imbalance)

This notebook demonstrates:

1. Baseline CNN model
2. Class weighting
3. Oversampling
4. Data augmentation
5. Focal loss

Each method is evaluated using:
- Validation Accuracy
- Test Accuracy
- Classification Report

Finally, results are compared in a summary table.

## Imports

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import pandas as pd
from tqdm import tqdm


## Dataset Path

Dataset should be structured as:

```
dataset/
   benign/
   malignant/
   normal/
```

In [3]:
DATASET_PATH = "dataset"

## Basic Transforms

In [4]:
basic_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

aug_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

## Load Dataset

In [7]:
import os
from PIL import Image
from torch.utils.data import Dataset

class BUSIDataset(Dataset):

    def __init__(self, root, transform=None):
        self.transform = transform
        self.images = []
        self.labels = []

        classes = sorted(os.listdir(root))

        for label, cls in enumerate(classes):
            folder = os.path.join(root, cls)

            for file in os.listdir(folder):

                if "_mask" in file:
                    continue

                path = os.path.join(folder, file)

                self.images.append(path)
                self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img = Image.open(self.images[idx]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = self.labels[idx]

        return img, label

**Stratified Splits**

In [8]:
from sklearn.model_selection import train_test_split
import numpy as np

dataset = BUSIDataset(DATASET_PATH, transform=basic_transform)

X = np.arange(len(dataset))
y = np.array(dataset.labels)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
) 
# ------------
# 70% train
# 15% validation
# 15% test

**DataLoaders**

In [9]:
from torch.utils.data import Subset
from torch.utils.data import DataLoader

train_dataset = Subset(dataset, X_train)
val_dataset = Subset(dataset, X_val)
test_dataset = Subset(dataset, X_test)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

**Verigying Class Distributions**

In [17]:
from collections import Counter

def print_counts(name, labels):
    counts = Counter(labels)
    print(name)
    for cls, count in sorted(counts.items()):
        print(f"  Class {cls}: {count}")
    print()

print_counts("Train", y_train)
print_counts("Val", y_val)
print_counts("Test", y_test)

Train
  Class 0: 306
  Class 1: 147
  Class 2: 93

Val
  Class 0: 65
  Class 1: 32
  Class 2: 20

Test
  Class 0: 66
  Class 1: 31
  Class 2: 20



## Model (Transfer Learning - ResNet18)

In [18]:
def get_model():
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, 3)
    return model

## Training Function

In [19]:
def train_model(model, train_loader, val_loader, criterion, epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        model.train()
        for images, labels in tqdm(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    return model

## Evaluation Function

In [20]:
def evaluate(model, loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    preds = []
    labels_list = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            pred = torch.argmax(outputs, axis=1).cpu().numpy()

            preds.extend(pred)
            labels_list.extend(labels.numpy())

    acc = accuracy_score(labels_list, preds)
    report = classification_report(labels_list, preds, output_dict=True)

    return acc, report

# 1️⃣ Baseline Model

In [21]:
model_baseline = get_model()
criterion = nn.CrossEntropyLoss()

model_baseline = train_model(model_baseline, train_loader, val_loader, criterion)

val_acc_base, val_report_base = evaluate(model_baseline, val_loader)
test_acc_base, test_report_base = evaluate(model_baseline, test_loader)

print("Baseline Test Accuracy:", test_acc_base)

C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 35/35 [00:06<00:00,  5.47it/s]


Baseline Test Accuracy: 0.8632478632478633


# 2️⃣ Class Weighting

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [25]:
targets = [label for _, label in dataset]
class_counts = np.bincount(targets)

weights = 1. / class_counts
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

criterion_weighted = nn.CrossEntropyLoss(weight=class_weights)

model_weighted = get_model().to(device)
model_baseline = model_baseline
model_weighted = train_model(model_weighted, train_loader, val_loader, criterion_weighted)

val_acc_weight, _ = evaluate(model_weighted, val_loader)
test_acc_weight, _ = evaluate(model_weighted, test_loader)

C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 35/35 [00:06<00:00,  5.57it/s]


# 3️⃣ Oversampling

In [26]:
class_sample_count = np.bincount(targets)
weights = 1. / class_sample_count
samples_weight = weights[targets]

sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

oversample_loader = DataLoader(dataset, batch_size=16, sampler=sampler)

model_over = get_model()
criterion = nn.CrossEntropyLoss()

model_over = train_model(model_over, oversample_loader, val_loader, criterion)

val_acc_over, _ = evaluate(model_over, val_loader)
test_acc_over, _ = evaluate(model_over, test_loader)

C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 49/49 [00:09<00:00,  5.08it/s]


# 4️⃣ Focal Loss

In [27]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

criterion_focal = FocalLoss()

model_focal = get_model()
model_focal = train_model(model_focal, train_loader, val_loader, criterion_focal)

val_acc_focal, _ = evaluate(model_focal, val_loader)
test_acc_focal, _ = evaluate(model_focal, test_loader)

C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 35/35 [00:06<00:00,  5.45it/s]


# 📊 Final Comparison

In [28]:
results = pd.DataFrame({
    "Method": [
        "Baseline",
        "Class Weighting",
        "Oversampling",
        "Focal Loss"
    ],
    "Validation Accuracy": [
        val_acc_base,
        val_acc_weight,
        val_acc_over,
        val_acc_focal
    ],
    "Test Accuracy": [
        test_acc_base,
        test_acc_weight,
        test_acc_over,
        test_acc_focal
    ]
})

results

,Method,Validation Accuracy,Test Accuracy
0,Baseline,0.854701,0.863248
1,Class Weighting,0.880342,0.837607
2,Oversampling,0.991453,0.991453
3,Focal Loss,0.854701,0.846154
